# Lyric Engine — Kaggle Training Notebook

**Runtime:** P100 GPU (16GB VRAM, 29GB RAM — free)
**What this does:** Fine-tunes Mistral 7B on genre-tagged lyrics with phoneme annotations.

Stages:
1. Setup & install
2. Clone repo
3. Config
4. Load lyrics dataset
5. Build style vectors
6. Verify pipeline
7. Load model
8. Train Stage 2 genre LoRA
9. Test generation
10. Save checkpoint

In [ ]:
# -- 1. Check GPU --
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — enable GPU in Settings"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')
!nvidia-smi

In [ ]:
# -- 2. Install dependencies --
!pip install -q transformers peft accelerate bitsandbytes trl datasets \
    pronouncing sentence-transformers textblob \
    fastapi uvicorn rich typer
print('Done.')

In [ ]:
# -- 3. Clone / update repo --
import os
REPO = 'https://github.com/SMXFREEZE/lyric-engine'
DEST = '/kaggle/working/lyric-engine'

if os.path.exists(f'{DEST}/.git'):
    print('Repo already cloned — pulling latest...')
    !git -C {DEST} pull origin main
else:
    print('Cloning repo...')
    !git clone {REPO} {DEST}

%cd {DEST}
!git log --oneline -5   # shows last 5 commits so you know what version is running

import sys
sys.path.insert(0, '.')

CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/style_vectors', exist_ok=True)
print(f'Checkpoints will save to: {CHECKPOINT_DIR}')

In [ ]:
# -- 4. Config --
import os

# Add your HuggingFace token if using Llama (not needed for Mistral)
HF_TOKEN = ''   # huggingface.co/settings/tokens
if HF_TOKEN:
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

BASE_MODEL = 'mistralai/Mistral-7B-v0.1'   # fits on P100 (16GB VRAM)
# BASE_MODEL = 'meta-llama/Llama-3.1-8B'  # needs HF approval + token

GENRE      = 'trap'
BATCH_SIZE = 2     # P100 has 16GB — can handle batch 2
GRAD_ACCUM = 16    # effective batch = 32
use_4bit   = True

print(f'Base model : {BASE_MODEL}')
print(f'Genre      : {GENRE}')
print(f'Eff. batch : {BATCH_SIZE * GRAD_ACCUM}')

In [ ]:
# -- 5. Load lyrics from HuggingFace (no API key needed) --
import json
from datasets import load_dataset

LINES_PER_SONG   = 40   # lines per synthetic song
SONGS_PER_ARTIST = 5    # how many songs per synthetic artist

print('Loading rap lyrics dataset (~47 MB)...')
ds = load_dataset('Cropinky/rap_lyrics_english', split='train')
print(f'Total lines: {len(ds):,}')

all_lines = [row['text'] for row in ds if row['text'].strip()]
songs = []
cap = min(len(all_lines), LINES_PER_SONG * 3000)
chunk_idx = 0
for i in range(0, cap, LINES_PER_SONG):
    chunk = all_lines[i : i + LINES_PER_SONG]
    if len(chunk) < 8:
        continue
    artist_id = chunk_idx // SONGS_PER_ARTIST
    songs.append({
        'artist': f'artist_{artist_id:04d}',
        'title':  f'song_{chunk_idx:04d}',
        'genre':  GENRE,
        'lyrics': '\n'.join(chunk),
    })
    chunk_idx += 1

out_path = f'data/raw/{GENRE}.jsonl'
with open(out_path, 'w') as f:
    for s in songs:
        f.write(json.dumps(s) + '\n')

n_artists = len(set(s['artist'] for s in songs))
print(f'Saved {len(songs)} songs from {n_artists} artists to {out_path}')

In [ ]:
# -- 6. Build style vectors --
from src.data.style_extractor import build_style_vectors

print('Building artist style vectors...')
style_vectors = build_style_vectors(
    jsonl_path=f'data/raw/{GENRE}.jsonl',
    out_dir='data/style_vectors',
    min_songs=3,
)
print(f'Style vectors built for {len(style_vectors)} artists')

In [ ]:
# -- 7. Verify pipeline --
from src.data.phoneme_annotator import annotate_line
from src.data.rhyme_labeler import detect_scheme
from src.data.valence_scorer import score_lyrics

with open(f'data/raw/{GENRE}.jsonl') as f:
    sample = json.loads(f.readline())

lines = sample['lyrics'].splitlines()[:8]
print(f"Artist: {sample['artist']} | Title: {sample['title']}")

scheme = detect_scheme(lines)
print(f"Rhyme scheme: {scheme['scheme_str']} ({scheme['scheme_type']})")
print(f"Rhyme density: {scheme['rhyme_density']}")

for em in score_lyrics('\n'.join(lines)):
    ann = annotate_line(em.text)
    print(f"  [{em.valence:+.2f}v {em.arousal:.2f}a] [{ann.total_syllables}syl] {em.text[:60]}")

In [ ]:
# -- 8. Load base model --
from src.model.lyrics_model import load_base_model, apply_lora, LyricsModel

device = 'cuda'

base, tokenizer = load_base_model(BASE_MODEL, use_4bit=use_4bit, device=device)
base = apply_lora(base, rank=16, alpha=32)
model = LyricsModel(base, d_model=base.config.hidden_size)

base.print_trainable_parameters()
print(f'Model on: {next(model.parameters()).device}')

In [ ]:
# -- 9. Train Stage 2 genre LoRA --
from src.training.sft import train_sft

train_sft(
    stage=2,
    genre=GENRE,
    data_path=f'data/raw/{GENRE}.jsonl',
    base_model=BASE_MODEL,
    output_dir=CHECKPOINT_DIR,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM,
    epochs=1,
    lr=2e-4,
    lora_rank=16,
    use_4bit=use_4bit,
    style_vec_dir='data/style_vectors',
)

In [ ]:
# -- 10. Test generation --
from transformers import AutoModelForCausalLM, AutoTokenizer as HFTok
from peft import PeftModel
from src.inference.engine import LyricsEngine, SongMemory

ckpt_path = f'{CHECKPOINT_DIR}/genre_{GENRE}/epoch_1'

tok = HFTok.from_pretrained(ckpt_path)
tok.pad_token = tok.eos_token

base_mdl = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, load_in_4bit=True, device_map='auto'
)
trained_mdl = PeftModel.from_pretrained(base_mdl, ckpt_path)

engine = LyricsEngine(trained_mdl, tok, device=device, beam_size=8)
memory = SongMemory(genre=GENRE, rhyme_scheme='AABB', target_syllables=10)
memory.sections.append(('[BUILD]', 'VERSE'))

print(f'\n=== Generated {GENRE.upper()} verse ===')
verse = engine.generate_verse(memory, num_lines=8, section='VERSE', arc_token='[BUILD]')
for line in verse:
    print(f'  {line}')

In [ ]:
# -- 11. Checkpoint location --
# Kaggle saves /kaggle/working/ automatically after the session ends.
# Download the checkpoint from: Output tab on the right side of this notebook.
import os
ckpt = f'{CHECKPOINT_DIR}/genre_{GENRE}/epoch_1'
print(f'Checkpoint saved at: {ckpt}')
print('Files:')
for f in os.listdir(ckpt):
    size = os.path.getsize(f'{ckpt}/{f}')
    print(f'  {f:40s} {size/1e6:.1f} MB')